# 데이터 불러오기

In [26]:
import os
from teds_tensor_dataset import TEDSTensorDataset
from utils.processing_utils import train_test_split_customed
from torch.utils.data import DataLoader

cur_dir = os.getcwd()
root = os.path.join(cur_dir, 'data_tensor_cache')
dataset = TEDSTensorDataset(root)
_, _, test_dataloader = train_test_split_customed(dataset=dataset,
                          batch_size=32)


저장되어 있는 전처리된 데이터가 있습니다. 해당 데이터를 불러오는 중..
불러오기 완료
Train Set Size: 975896
Valid Set Size: 209119
Test Set Size: 209123


# 모델 불러오기

## 래퍼 모델을 만들어야 한다고 함

In [27]:
import os
import sys
import torch
import torch.nn as nn
from torch_geometric.nn import GINConv
from torch.nn import GRU
from torch.nn.utils.rnn import PackedSequence, pack_padded_sequence
from models.entity_embedding import EntityEmbeddingBatch3

def get_mask(los_batch: torch.Tensor):
    '''
    주어진 배치별 각 케이스의 시간 길이 (sequence length) 정보를 담고 있는 텐서를 사용하여, 
    해당 길이에 해당하는 부분은 1로, 나머지 부분은 0으로 채워진 마스크 텐서를 생성
    '''
    max_los = 37

    indices = torch.arange(max_los, device=los_batch.device) 
    mask = (indices < los_batch.unsqueeze(1))

    return mask.int().unsqueeze(2)


def to_temporal_gingru(x: torch.Tensor, los_batch: torch.Tensor):
    # process:  [batch, T, gin_h_dim]으로 변환하기 (gin_h_dim = F*num_layers)
    # 1. [batch, 2, gin_h_dim]으로 변환하기 --> .reshape(-1, 2, F)
    # 2. [batch, T, gin_h_dim]으로 변환하기
    # process: PackedSequence로 변환하기

    batch_size = int(x.shape[0] / 2) # [batch(twiced), f]

    ad = x[:batch_size].unsqueeze(1) # 브로드캐스팅을 위해 [B, 1, F]
    dis = x[batch_size:]
    
    # los_batch를 가지고 0,1 마스킹 행렬 생성
    mask = get_mask(los_batch=los_batch) # [B, 37, 1]
    padded = ad * mask # [B, 1, F] * [B, 37, 1] --> [B, 37, F] @@ 브로드캐스팅 !! @@

    batch_indices = torch.arange(batch_size, device=ad.device)
    last_time_indices = los_batch - 1

    padded[batch_indices, last_time_indices, :] = dis

    # -----PackedSequence 만들기-----
    lengths = los_batch.cpu() 

    # 내림차순(descending=True)으로 정렬
    lengths_sorted, sorted_indices = torch.sort(lengths, descending=True)

    padded_sorted = padded[sorted_indices]

    packed_data = pack_padded_sequence(
        padded_sorted,
        lengths_sorted,
        batch_first=True,
        enforce_sorted=True
    )

    return packed_data, sorted_indices # padded_sorted의 디바이스를 따라감


def seperate_x(x: torch.Tensor, ad_idx_t, dis_idx_t, device):
    
    ad_tensor =torch.index_select(x, dim=1, index=ad_idx_t) # [B, 60, F]
    dis_tensor = torch.index_select(x, dim=1, index=dis_idx_t) # [B, 60, F]

    return torch.concatenate((ad_tensor, dis_tensor), dim=0) # [B*2, 60, F]



class ExplainerCompatibleGinGru(nn.Module):
    def __init__(self, batch_size, col_list, col_dims, ad_col_index, dis_col_index, embedding_dim, gin_hidden_channel, train_eps, gin_layers, gru_hidden_channel):
        '''
        Args:
            col_info(list): [col_dims, col_list]
                            col_list(list): 데이터에서 나타나는 변수의 순서
                            col_dims(list): 각 변수 별 범주의 개수, 순서는 col_list를 따라야 함
            embedding_dim(int): 엔티티 임베딩 후의 차원
            gin_hidden_channel(int): GIN의 hidden channel, 일반적으로 인풋, 히든, 아웃풋 차원을 동일하게 설정하는 것이 흔하고 효율적임
            train_eps(bool): GIN의 epsilon을 훈련할 것인지, 고정할 것인지

                                Gemini의 당부
                                다만, 모델을 로드하기 위해 새 인스턴스를 만들 때(e.g., model = MyGINModel(...)), 
                                해당 모델이 epsilon 슬롯을 가지고 있도록 반드시 train_eps=True를 동일하게 설정하여 초기화해야 합니다. 
                                그렇지 않으면 저장된 state_dict와 새 모델의 구조가 일치하지 않아 로딩에 실패합니다.

            gin_layers(int): GIN의 레이어 개수
            gru_hidden_channel(int): GRU의 hidden channel, 이는 모델의 기억 용량을 의미함
                                     우리 데이터는 크게 설정할 필요가 없을 것으로 보임 (오직 1번 달라지기 때문)
                                     시간축으로만 보면 그렇지만, 
                                     GIN이 출력하는 특징의 복잡도를 수용할 수 있는 적절한 최소 크기(Medium-sized*로 설정하는 것이 가장 좋다.
                                     gin_hidden_channel과 동일하게 설정하여 시작하는 것이 좋음
        '''
        super().__init__()
        self.batch_size = batch_size
        self.col_dims = col_dims
        self.col_list = col_list
        self.hidden_channel = gin_hidden_channel

        self.ad_idx_t = torch.tensor(ad_col_index)
        self.dis_idx_t = torch.tensor(dis_col_index)

        # EntityEmbedding 레이어 정의
        self.entity_embedding_layer = EntityEmbeddingBatch3(col_dims=self.col_dims, embedding_dim=embedding_dim)
        
        # GIN 레이어 정의
        # MLP 구조는 GIN 논문의 권장 사항(2-Layer MLP, 배치 정규화 )
        gin_nn_input = nn.Sequential(
             nn.Linear(embedding_dim, gin_hidden_channel),
             nn.LayerNorm(gin_hidden_channel),
             nn.ReLU(),

             nn.Linear(gin_hidden_channel, gin_hidden_channel) # 논문에서 적용된 배치 정규화 
             # nn.LayerNorm(h_dim),  # 마지막 레이어 이후에는 선택적
        )

        gin_nn = nn.Sequential(
             nn.Linear(gin_hidden_channel, gin_hidden_channel),
             nn.LayerNorm(gin_hidden_channel),
             nn.ReLU(),

             nn.Linear(gin_hidden_channel, gin_hidden_channel) # 논문에서 적용된 배치 정규화 
             # nn.LayerNorm(h_dim),  # 마지막 레이어 이후에는 선택적
        )

        self.gin_layers = nn.ModuleList()

        gin_layer1 = GINConv(nn=gin_nn_input, eps=0, train_eps=train_eps)
        self.gin_layers.append(gin_layer1)
        
        for _ in range(gin_layers - 1):
            gin_layer_hidden = GINConv(nn=gin_nn, eps=0, train_eps=train_eps)
            self.gin_layers.append(gin_layer_hidden)
        
        gru_input_ch = gin_hidden_channel * gin_layers
        self.gru_layer = GRU(input_size=gru_input_ch, hidden_size=gru_hidden_channel)

        # 분류기 레이어 정의
        self.classifier_b = nn.Sequential(
            nn.Linear(gru_hidden_channel, gru_hidden_channel * 2),
            nn.ReLU(),
            nn.Linear(gru_hidden_channel * 2, 1)
        )

    def forward(self, x_embedded, template_edge_index, **kwargs):
        '''
        template_edge_index: supersized edge_index
        **kwargs: LOS_batch, device
        '''
        device = kwargs.get("device")
        LOS_batch = kwargs.get("LOS_batch")

        # x_embedded shape: [72, 32]
        x_embedded = x_embedded.unsqueeze(0)
        zero_tensor = torch.zeros([31, 72, 32])
        gin_input = torch.concatenate([x_embedded, zero_tensor], dim=0) # gin_input shape: [32, 72, 32]
        
        
        self.ad_idx_t = self.ad_idx_t.to(device)
        self.dis_idx_t = self.dis_idx_t.to(device)

        # process: [batch * 2, num_nodes, feature_dim]으로 변환하기
        # 이때 시간 축 평탄화는 다음과 같이 되어야 함: 1, 2, 1, 2, 1, 2, ...
        # 위와 같이 될 필요는 없음: 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, ...
        # 이렇게 되어도 문제는 없다.
        x_seperated = seperate_x(x=gin_input, 
                                 ad_idx_t=self.ad_idx_t, 
                                 dis_idx_t=self.dis_idx_t, 
                                 device=device)

        # GIN에 입력
        x_after_gin = x_seperated
        sum_pooled = []
        for layer in self.gin_layers:
            x_after_gin = layer(x_after_gin, template_edge_index) # [B * 2, N(60), F(32)]
            x_sum = torch.sum(x_after_gin, dim=1) # [B * 2, N(60), F(32)] --> [B * 2, F(32)]
            sum_pooled.append(x_sum)
        gin_result = torch.concatenate(sum_pooled, dim=1) # [B*2, F*num_gin_layers]
            
        # [B*2, F*num_gin_layers] --> [B, 37, F*num_gin_layers]
        temporal_embedding, sorted_indices = to_temporal_gingru(x=gin_result, los_batch=LOS_batch)

        # GRU에 입력
        gru_out, gru_h = self.gru_layer(temporal_embedding)

        gru_h = gru_h.squeeze(0)

        # PackedSequence로 만들었기 때문에 (시간 길이 순 정렬) 역정렬해야 함
        inv_indices = torch.argsort(sorted_indices.to(gru_h.device))
        gru_h = gru_h[inv_indices]                           

        # Classifier에 입력
        classifier_result = self.classifier_b(gru_h)
        return classifier_result[0]

In [28]:
batch_size = 32
embedding_dim = 32
gin_hidden_channel = 32

col_list, col_dims, ad_col_index, dis_col_index = dataset.col_info

model = ExplainerCompatibleGinGru(
    batch_size=batch_size,
    col_list=col_list,
    col_dims=col_dims,
    ad_col_index=ad_col_index,
    dis_col_index=dis_col_index,
    embedding_dim=embedding_dim,
    gin_hidden_channel=gin_hidden_channel,
    train_eps=True,
    gin_layers=2,
    gru_hidden_channel=64
)
model.eval()

ExplainerCompatibleGinGru(
  (entity_embedding_layer): EntityEmbeddingBatch3(
    (embedding_layer): Embedding(1143, 32)
  )
  (gin_layers): ModuleList(
    (0-1): 2 x GINConv(nn=Sequential(
      (0): Linear(in_features=32, out_features=32, bias=True)
      (1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Linear(in_features=32, out_features=32, bias=True)
    ))
  )
  (gru_layer): GRU(64, 64)
  (classifier_b): Sequential(
    (0): Linear(in_features=64, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [29]:
def load_checkpoint(model, optimizer, scheduler, filename, device='cpu'):
    """
    저장된 체크포인트를 불러와 model, optimizer, scheduler 상태를 복원한다.
    반환값: (start_epoch, best_loss)
    """
    checkpoint = torch.load(filename, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    start_epoch = checkpoint['epoch'] + 1  # 다음 epoch부터 재시작
    best_loss = checkpoint['best_loss']

    return start_epoch, best_loss


In [30]:
import torch
from torch.optim.lr_scheduler import ReduceLROnPlateau

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, "min", patience=10)

filename = os.path.join(cur_dir, 'checkpoints', 'gingru', 'real', 'best_gingru_epoch_11_loss_0.3812.pth')
start_epoch, best_loss = load_checkpoint(model=model, # 외부 변수이 model에 대해 알아서 값이 변경된다. 힙 메모리에 있는 변수 (객체 생성)
                                         optimizer=optimizer,
                                         scheduler=scheduler,
                                         filename=filename)

# Explainer 정의

In [6]:
from torch_geometric.explain import Explainer, GNNExplainer
from torch_geometric.explain import ModelConfig, ThresholdConfig

# Hyperparameters
epochs = 100
lr = 0.01
explanation_type = 'model' # can be phenomenon
node_mask_type = None
edge_mask_type = 'object'
model_config = ModelConfig(mode='binary_classification',
                           task_level='graph',
                           return_type='raw')

threshold_config = None # 서브그래프를 알고자 한다면 ThresholdConfig 사용하기

In [7]:
from utils.processing_utils import mi_edge_index_batched
# 첫 번째 데이터를 가져와 예시로 돌려보기 
device = torch.device("cpu")

batch1 = next(test_dataloader.__iter__())
x_label = batch1[0].to(device).detach()

x_emb = model.entity_embedding_layer(x_label)
target = batch1[1].to(device)
los_batch = batch1[2].to(device)

# 엣지 인덱스 정의
mi_dict_path = os.path.join(root, 'data', 'mi_dict_static.pickle')
edge_index = mi_edge_index_batched(batch_size=1,
                                        mi_dict_path=mi_dict_path,
                                        top_k=6,
                                        return_edge_attr=False)

x = x_emb[0].detach()
target = target.detach()
los_batch = los_batch.detach()

In [20]:
# 1. 알고리즘 설정 수정 (Sparsity 제거)
explainer_algorithm = GNNExplainer(
    epochs=300,        # 충분히 학습하도록 증가
    lr=0.05,           # 학습률 증가
    edge_size=0.0,     # 👈 [중요] 0.0으로 설정
    edge_mask_type='object',
    node_mask_type=None
)

explainer = Explainer(model=model,
                      algorithm=explainer_algorithm,
                      explanation_type=explanation_type,
                      model_config=model_config,
                      node_mask_type=node_mask_type,
                      edge_mask_type=edge_mask_type,
                      threshold_config=threshold_config)

# 2. Explainer 다시 연결
explainer.algorithm = explainer_algorithm

# 3. 단일 그래프용 edge_index 사용 (이미 수정하셨다면 패스)
single_edge_index = mi_edge_index_batched(batch_size=1, mi_dict_path=mi_dict_path, top_k=6, return_edge_attr=False).to(device)

# 4. 실행
explanation = explainer.algorithm.forward(
    model=model,
    x=x,
    edge_index=single_edge_index, # 단일 엣지 인덱스
    target=target,
    index=0,
    LOS_batch=los_batch,
    device=device
)

print("결과 확인:")
print(explanation.edge_mask.max()) # 0이 아닌 값이 나오는지 확인
print(explanation.edge_mask.topk(5))

결과 확인:
tensor(0.)
torch.return_types.topk(
values=tensor([0., 0., 0., 0., 0.]),
indices=tensor([2, 4, 0, 1, 3]))


In [14]:
algorithm = GNNExplainer(epochs=epochs, lr=lr, edge_size=0.0)

explainer = Explainer(model=model,
                      algorithm=algorithm,
                      explanation_type=explanation_type,
                      model_config=model_config,
                      node_mask_type=node_mask_type,
                      edge_mask_type=edge_mask_type,
                      threshold_config=threshold_config)

explanation = explainer.algorithm.forward(
    model=model,
    x=x,
    edge_index=edge_index,
    target=target,
    index=0,
    LOS_batch=los_batch,
    device=device
)

In [15]:
# 2. 훈련된 결과 확인 (가장 중요한 부분)
print("설명 생성 완료!")
print(f"Edge Mask Shape: {explanation.edge_mask.shape}") # [Num_Edges]
print(f"가장 중요한 엣지 상위 5개 점수: {explanation.edge_mask.topk(5).values}")

설명 생성 완료!
Edge Mask Shape: torch.Size([476])
가장 중요한 엣지 상위 5개 점수: tensor([0., 0., 0., 0., 0.])


In [16]:
# 엣지 중요도 가져오기
edge_mask = explanation.edge_mask.detach() # 혹시 모를 grad 제거
num_nodes = x.size(0)

# 노드 중요도 계산 (엣지 합산 방식)
node_importance = torch.zeros(num_nodes, device=device)
node_importance.index_add_(0, edge_index[0], edge_mask) # 출발 노드에 합산
node_importance.index_add_(0, edge_index[1], edge_mask) # 도착 노드에 합산

# 최종 결과 출력
top_nodes = node_importance.topk(5)
print(f"\n최종적으로 가장 중요한 노드 5개: {top_nodes.indices.tolist()}")


최종적으로 가장 중요한 노드 5개: [0, 1, 2, 3, 4]


In [11]:
for idx in top_nodes.indices:
    print(col_list[idx])

STFIPS
EDUC
MARSTAT
SERVICES
DETCRIM


In [12]:
for idx in node_importance.sort(descending=True).indices:
    print(col_list[idx])

STFIPS
EDUC
MARSTAT
SERVICES
DETCRIM
PSOURCE
NOPRIOR
ARRESTS
EMPLOY
METHUSE
PSYPROB
PREG
GENDER
VET
LIVARAG
DAYWAIT
SERVICES_D
EMPLOY_D
LIVARAG_D
ARRESTS_D
DSMCRIT
AGE
RACE
ETHNIC
DETNLF
DETNLF_D
PRIMINC
SUB1
SUB2
SUB3
SUB1_D
SUB2_D
SUB3_D
ROUTE1
ROUTE2
ROUTE3
FREQ1
FREQ2
FREQ3
FREQ1_D
FREQ2_D
FREQ3_D
FRSTUSE1
FRSTUSE2
FRSTUSE3
HLTHINS
PRIMPAY
FREQ_ATND_SELF_HELP
FREQ_ATND_SELF_HELP_D
ALCFLG
COKEFLG
MARFLG
HERFLG
METHFLG
OPSYNFLG
PCPFLG
HALLFLG
MTHAMFLG
AMPHFLG
STIMFLG
BENZFLG
TRNQFLG
BARBFLG
SEDHPFLG
INHFLG
OTCFLG
OTHERFLG
DIVISION
REGION
IDU
ALCDRUG
CBSA2020


In [13]:
# 전체 테스트 데이터셋에 대한 함수
# 는 무리이고 샘플을 뽑아서 학습한다.
from torch_geometric.explain.explainer import Explainer
from torch_geometric.explain.algorithm import GNNExplainer, DummyExplainer

algorithm = GNNExplainer(epochs=epochs, lr=lr)

explainer = Explainer(model=model,
                      algorithm=algorithm,
                      explanation_type=explanation_type,
                      model_config=model_config,
                      node_mask_type=node_mask_type,
                      edge_mask_type=edge_mask_type,
                      threshold_config=threshold_config)

from tqdm import tqdm
sample_size = 2
counter = 0

model = model.to(device)

# tqdm을 통해 예상 시간을 정확하게 알기 위해 미리 리스트 형태로 저장
batch_list = []
counter = 0
for batch in test_dataloader:
    if counter == sample_size: break
    batch_list.append(batch)
    counter += 1


aggregated_node_importance = torch.zeros(72, device=device)
per_los_dic = {i: torch.zeros(72, device=device) for i in range(1,38)}

for batch in tqdm(batch_list):
    x_label = batch[0]
    target = batch[1]
    los_batch = batch[2]

    model.eval()

    with torch.no_grad():
        x_emb = model.entity_embedding_layer(x_label)
    
    x = x_emb[0].detach()
    target = target.detach()
    los_batch = los_batch.detach()

    explanation = explainer.algorithm.forward(
        model=model,
        x=x,
        edge_index=edge_index,
        target=target,
        index=0,
        LOS_batch=los_batch,
        device=device
    )

    # 엣지 중요도 가져오기
    edge_mask = explanation.edge_mask.detach() # 혹시 모를 grad 제거
    num_nodes = x.size(0)

    # 노드 중요도 계산 (엣지 합산 방식)
    curr_node_importance = torch.zeros(num_nodes, device=device)
    curr_node_importance.index_add_(0, edge_index[0], edge_mask) # 출발 노드에 합산
    curr_node_importance.index_add_(0, edge_index[1], edge_mask) # 도착 노드에 합산
    
    # 누적 합
    aggregated_node_importance += curr_node_importance
    per_los_dic[int(los_batch[0])] += curr_node_importance
avg_node_importance = aggregated_node_importance / sample_size

print(f"\n 총 {sample_size}개 샘플에 대한 평균 노드 중요도 Top 5")
top_vals, top_indices = avg_node_importance.topk(5)
for idx in top_indices:
    print(f"{col_list[idx]} : {avg_node_importance[idx].item():.4f}")


100%|██████████| 2/2 [00:05<00:00,  2.53s/it]


 총 2개 샘플에 대한 평균 노드 중요도 Top 5
STFIPS : 0.0000
EDUC : 0.0000
MARSTAT : 0.0000
SERVICES : 0.0000
DETCRIM : 0.0000
